# 04 - Modelado multibacteria V2 binario

Modelado para predecir:

- `Susceptible`
- `Resistant`

Metrica principal:

- `f1_macro`

Metricas auxiliares:

- `balanced_accuracy`
- `accuracy`
- `mse` sobre clases codificadas
- sensibilidad/especificidad para la clase clinica `Resistant`
- ROC-AUC y PR-AUC cuando el modelo entrega probabilidades

Esta version ejecuta una busqueda ampliada: 14 familias de modelos con 200 configuraciones por familia. En total son 2800 combinaciones de hiperparametros y, con validacion cruzada de 3 folds, 8400 entrenamientos internos estimados.

Nota metodologica: no existe probar "todos" los hiperparametros posibles porque muchos son continuos o tienen rangos practicamente infinitos. Lo defendible es una busqueda amplia, reproducible y con rangos clinica/tecnicamente razonables.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    mean_squared_error, mean_absolute_error, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, average_precision_score, log_loss,
    brier_score_loss, matthews_corrcoef, cohen_kappa_score, fbeta_score,
    roc_curve, precision_recall_curve, auc,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier,
    AdaBoostClassifier, GradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

RANDOM_STATE = 42
DATASET_USAR = "balanceado_organismo_clase"
USAR_CUDA_XGBOOST = True
CONFIGURACIONES_POR_MODELO = 200

RUTA_ACTUAL = Path.cwd().resolve()
RUTA_MODELO = None
for candidata in [RUTA_ACTUAL, *RUTA_ACTUAL.parents]:
    if (candidata / "V2" / "DATOS_PROCESADOS").exists() and candidata.name.lower() == "modelo":
        RUTA_MODELO = candidata
        break
    if (candidata / "modelo" / "V2" / "DATOS_PROCESADOS").exists():
        RUTA_MODELO = candidata / "modelo"
        break
if RUTA_MODELO is None:
    raise FileNotFoundError("No se encontro modelo/V2/DATOS_PROCESADOS")

RUTA_V2 = RUTA_MODELO / "V2"
RUTA_PROCESADOS = RUTA_V2 / "DATOS_PROCESADOS"
RUTA_RESULTADOS = RUTA_V2 / "05_RESULTADOS"
RUTA_GRAFICAS = RUTA_V2 / "GRAFICAS"
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
RUTA_GRAFICAS.mkdir(parents=True, exist_ok=True)

rutas_dataset = {
    "completo": RUTA_PROCESADOS / "07_dataset_v2_multibacteria_completo.csv",
    "balanceado_clase": RUTA_PROCESADOS / "08_dataset_v2_multibacteria_balanceado_clase.csv",
    "balanceado_organismo_clase": RUTA_PROCESADOS / "09_dataset_v2_multibacteria_balanceado_organismo_clase.csv",
}
RUTA_DATASET = rutas_dataset[DATASET_USAR]
print("Dataset:", RUTA_DATASET)


## 1. Cargar dataset y variables

In [2]:
df = pd.read_csv(RUTA_DATASET, low_memory=False)

ruta_decisiones = RUTA_PROCESADOS / "17_decision_variables_v2.csv"
if ruta_decisiones.exists():
    decisiones = pd.read_csv(ruta_decisiones)
    variables_excluidas = set(decisiones.loc[decisiones["decision"].eq("excluir"), "variable"])
else:
    variables_excluidas = set()

columnas_exp_prev = [c for c in df.columns if c.startswith("exp_prev_")]
if not columnas_exp_prev:
    raise ValueError(
        "El dataset V2 no tiene columnas exp_prev_*. Ejecuta primero el notebook 02; "
        "la exposicion antibiotica previa es obligatoria en esta version."
    )

columnas_excluir = {"anon_id", "pat_enc_csn_id_coded", "order_proc_id_coded", "order_time_jittered_utc", "susceptibility"}
columnas_excluir = columnas_excluir.union(variables_excluidas)

X = df.drop(columns=[c for c in columnas_excluir if c in df.columns])
y_texto = df["susceptibility"].astype(str)

le = LabelEncoder()
y = le.fit_transform(y_texto)

columnas_categoricas = X.select_dtypes(include=["object", "string"]).columns.tolist()
columnas_numericas = [c for c in X.columns if c not in columnas_categoricas]

print("Filas:", len(df))
print("Predictores:", X.shape[1])
print("Categoricas:", len(columnas_categoricas))
print("Numericas:", len(columnas_numericas))
print("Columnas de exposicion antibiotica previa exp_prev_*:", len(columnas_exp_prev))
print("Clases:", dict(zip(le.classes_, le.transform(le.classes_))))
print(y_texto.value_counts())


Filas: 242923
Predictores: 61
Categoricas: 6
Numericas: 55
Columnas de exposicion antibiotica previa exp_prev_*: 18
Clases: {'Resistant': np.int64(0), 'Susceptible': np.int64(1)}
susceptibility
Susceptible    150000
Resistant       92923
Name: count, dtype: int64


## 2. Preprocesamiento y split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

preprocesador_escalado = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

preprocesador_minmax = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

preprocesador_arboles = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), columnas_numericas),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="SIN_REGISTRO")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))]), columnas_categoricas),
])

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
pesos_train = compute_sample_weight(class_weight="balanced", y=y_train)
print("Train:", X_train.shape, "Test:", X_test.shape)


## 3. Modelos e hiperparametros

Se ejecuta una busqueda ampliada, no aleatoria, sobre 14 familias de modelos. La grilla cambia parametros con sentido tecnico: regularizacion, complejidad, profundidad, numero de estimadores, learning rate, hojas minimas, reduccion dimensional para modelos sensibles a dimensionalidad, vecinos para KNN, suavizado para Naive Bayes y margenes para SVM/SGD.

Advertencia practica: SVM con kernel RBF completo no se activa por defecto porque con cientos de miles de filas y one-hot encoding puede tardar dias o quedarse sin RAM. Se usa `LinearSVC`, que es la variante defendible para datasets grandes y dispersos.


In [ ]:
def seleccionar_configuraciones_uniformes(configuraciones, n=CONFIGURACIONES_POR_MODELO):
    """Selecciona configuraciones repartidas por toda la grilla, no solo las primeras."""
    if len(configuraciones) < n:
        raise ValueError(f"Cada modelo debe tener al menos {n} configuraciones, recibio {len(configuraciones)}")
    if len(configuraciones) == n:
        return configuraciones
    indices = np.linspace(0, len(configuraciones) - 1, n, dtype=int)
    return [configuraciones[i] for i in indices]


def construir_grid(configuraciones, esperado=CONFIGURACIONES_POR_MODELO):
    """Convierte configuraciones puntuales en param_grid para GridSearchCV.

    No existe una grilla infinita: se toma una cuota amplia y reproducible por modelo.
    Si una familia genera mas combinaciones, se eligen puntos repartidos por toda la grilla
    para cubrir valores bajos, medios y altos de los hiperparametros.
    """
    configuraciones = seleccionar_configuraciones_uniformes(configuraciones, esperado)
    return [{k: [v] for k, v in config.items()} for config in configuraciones]


def limitar_configuraciones(configuraciones, n=CONFIGURACIONES_POR_MODELO):
    """Mantiene la busqueda reproducible y comparable entre modelos."""
    return seleccionar_configuraciones_uniformes(configuraciones, n)


# Logistic regression: regularizacion L2 y elastic-net con solver saga.
config_lr = []
for C in np.geomspace(0.003, 100.0, 50):
    config_lr.append({"clf__C": C, "clf__fit_intercept": True, "clf__solver": "lbfgs", "clf__penalty": "l2", "clf__tol": 1e-4})
    config_lr.append({"clf__C": C, "clf__fit_intercept": False, "clf__solver": "lbfgs", "clf__penalty": "l2", "clf__tol": 1e-4})
    for l1_ratio in [0.15, 0.50, 0.85]:
        config_lr.append({"clf__C": C, "clf__fit_intercept": True, "clf__solver": "saga", "clf__penalty": "elasticnet", "clf__l1_ratio": l1_ratio, "clf__tol": 1e-3})

# SVM lineal: version escalable de SVM para dataset grande y disperso.
config_svm_linear = [
    {"clf__C": C, "clf__tol": tol, "clf__loss": loss, "clf__class_weight": class_weight, "clf__max_iter": 6000}
    for C in np.geomspace(0.005, 30.0, 25)
    for tol in [1e-4, 5e-4]
    for loss in ["hinge", "squared_hinge"]
    for class_weight in ["balanced", None]
]

# SGD: clasificador lineal rapido; prueba log-loss, modified-huber y hinge.
config_sgd = [
    {"clf__loss": loss, "clf__alpha": alpha, "clf__penalty": penalty, "clf__l1_ratio": l1_ratio, "clf__class_weight": "balanced", "clf__max_iter": 2500, "clf__tol": 1e-3}
    for loss in ["log_loss", "modified_huber", "hinge"]
    for alpha in np.geomspace(1e-6, 1e-2, 20)
    for penalty in ["l2", "elasticnet"]
    for l1_ratio in [0.15, 0.50]
]

# KNN: usa SVD para reducir dimensionalidad antes de calcular distancias.
config_knn = [
    {"svd__n_components": n_comp, "clf__n_neighbors": k, "clf__weights": weights, "clf__p": p}
    for n_comp in [30, 50, 80, 120, 160]
    for k in [3, 5, 7, 11, 15, 21, 31, 45, 61, 81]
    for weights in ["uniform", "distance"]
    for p in [1, 2]
]

# Naive Bayes complementario: funciona bien con matrices dispersas no negativas.
config_cnb = [
    {"clf__alpha": alpha, "clf__fit_prior": fit_prior, "clf__norm": norm}
    for alpha in np.geomspace(1e-4, 100.0, 50)
    for fit_prior in [True, False]
    for norm in [True, False]
]

# GaussianNB sobre representacion SVD densa.
config_gnb = [
    {"svd__n_components": n_comp, "clf__var_smoothing": var_smoothing}
    for n_comp in [30, 50, 80, 120]
    for var_smoothing in np.geomspace(1e-12, 1e-2, 50)
]

# Decision tree: profundidad, criterio, hoja minima y poda.
config_dt = [
    {"clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__criterion": criterion, "clf__ccp_alpha": ccp_alpha, "clf__min_samples_split": min_split}
    for max_depth in [4, 6, 8, 10, 12, 16, 20, 28, None]
    for min_leaf in [5, 10, 20, 40, 80, 120]
    for min_split in [2, 10]
    for criterion in ["gini", "entropy", "log_loss"]
    for ccp_alpha in [0.0, 0.0002, 0.001]
]

# Random forest: ensamble, profundidad, hoja minima, max_features y bootstrap.
config_rf = [
    {"clf__n_estimators": n_estimators, "clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__max_features": max_features, "clf__bootstrap": bootstrap}
    for n_estimators in [160, 240, 320, 420, 560]
    for max_depth in [12, 18, 24, None]
    for min_leaf in [2, 5, 10, 20, 40]
    for max_features in ["sqrt", "log2"]
    for bootstrap in [True, False]
]

# Extra Trees: similar a Random Forest, con cortes mas aleatorios.
config_extra_trees = [
    {"clf__n_estimators": n_estimators, "clf__max_depth": max_depth, "clf__min_samples_leaf": min_leaf, "clf__max_features": max_features, "clf__bootstrap": bootstrap}
    for n_estimators in [160, 240, 320, 420, 560]
    for max_depth in [12, 18, 24, None]
    for min_leaf in [2, 5, 10, 20, 40]
    for max_features in ["sqrt", "log2"]
    for bootstrap in [False, True]
]

# AdaBoost con arboles pequenos: explora boosting interpretable con stumps/arboles poco profundos.
config_adaboost = [
    {"clf__n_estimators": n_estimators, "clf__learning_rate": learning_rate, "clf__estimator__max_depth": depth, "clf__estimator__min_samples_leaf": min_leaf}
    for n_estimators in [80, 120, 180, 240, 320]
    for learning_rate in [0.01, 0.03, 0.05, 0.08, 0.12]
    for depth in [1, 2, 3, 4]
    for min_leaf in [10, 30, 60, 100]
]

# Gradient Boosting clasico: mas lento que HGB, pero sirve como comparacion.
config_gb = [
    {"clf__n_estimators": n_estimators, "clf__learning_rate": learning_rate, "clf__max_depth": depth, "clf__min_samples_leaf": min_leaf, "clf__subsample": subsample}
    for n_estimators in [80, 120, 180, 240]
    for learning_rate in [0.02, 0.04, 0.06, 0.10, 0.15]
    for depth in [2, 3, 4, 5]
    for min_leaf in [10, 30, 60]
    for subsample in [0.75, 0.90, 1.0]
]

# HistGradientBoosting: version eficiente para datasets grandes.
config_hgb = [
    {"clf__max_iter": max_iter, "clf__learning_rate": learning_rate, "clf__max_leaf_nodes": max_leaf_nodes, "clf__l2_regularization": l2, "clf__min_samples_leaf": min_leaf}
    for max_iter in [120, 180, 240, 320, 440]
    for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.12]
    for max_leaf_nodes in [15, 31, 63, 95]
    for l2 in [0.0, 0.1, 1.0, 2.0]
    for min_leaf in [20, 40]
]

# MLP: red neuronal pequena sobre SVD para evitar matriz one-hot densa gigante.
config_mlp = [
    {"svd__n_components": n_comp, "clf__hidden_layer_sizes": hidden, "clf__alpha": alpha, "clf__learning_rate_init": lr, "clf__activation": activation}
    for n_comp in [50, 80, 120, 160]
    for hidden in [(32,), (64,), (96,), (64, 32), (128, 64)]
    for alpha in [1e-5, 1e-4, 1e-3, 1e-2]
    for lr in [0.0005, 0.001, 0.003]
    for activation in ["relu", "tanh"]
]

# XGBoost: explora profundidad, learning rate, regularizacion y muestreo.
config_xgb = [
    {
        "clf__n_estimators": n_estimators,
        "clf__max_depth": max_depth,
        "clf__learning_rate": learning_rate,
        "clf__subsample": subsample,
        "clf__colsample_bytree": colsample,
        "clf__min_child_weight": min_child_weight,
        "clf__reg_lambda": reg_lambda,
        "clf__reg_alpha": reg_alpha,
    }
    for n_estimators in [160, 240, 320, 440, 600]
    for max_depth in [3, 4, 5, 6, 8]
    for learning_rate in [0.02, 0.03, 0.05, 0.08, 0.12]
    for subsample in [0.75, 0.85, 0.95]
    for colsample in [0.75, 0.85, 0.95]
    for min_child_weight in [1, 3, 5]
    for reg_lambda in [1.0, 2.0, 5.0]
    for reg_alpha in [0.0, 0.1]
]

modelos_parametros = {
    "logistic_regression": (
        Pipeline([("prep", preprocesador_escalado), ("clf", LogisticRegression(max_iter=2500, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE))]),
        construir_grid(config_lr),
    ),
    "svm_linear": (
        Pipeline([("prep", preprocesador_escalado), ("clf", LinearSVC(random_state=RANDOM_STATE, dual="auto"))]),
        construir_grid(config_svm_linear),
    ),
    "sgd_linear": (
        Pipeline([("prep", preprocesador_escalado), ("clf", SGDClassifier(random_state=RANDOM_STATE, n_jobs=-1))]),
        construir_grid(config_sgd),
    ),
    "knn_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", KNeighborsClassifier(algorithm="brute", n_jobs=-1))]),
        construir_grid(config_knn),
    ),
    "complement_naive_bayes": (
        Pipeline([("prep", preprocesador_minmax), ("clf", ComplementNB())]),
        construir_grid(config_cnb),
    ),
    "gaussian_naive_bayes_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", GaussianNB())]),
        construir_grid(config_gnb),
    ),
    "decision_tree": (
        Pipeline([("prep", preprocesador_arboles), ("clf", DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
        construir_grid(config_dt),
    ),
    "random_forest": (
        Pipeline([("prep", preprocesador_arboles), ("clf", RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1))]),
        construir_grid(config_rf),
    ),
    "extra_trees": (
        Pipeline([("prep", preprocesador_arboles), ("clf", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))]),
        construir_grid(config_extra_trees),
    ),
    "adaboost_tree": (
        Pipeline([("prep", preprocesador_arboles), ("clf", AdaBoostClassifier(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE), random_state=RANDOM_STATE))]),
        construir_grid(config_adaboost),
    ),
    "gradient_boosting": (
        Pipeline([("prep", preprocesador_arboles), ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
        construir_grid(config_gb),
    ),
    "hist_gradient_boosting": (
        Pipeline([("prep", preprocesador_arboles), ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE, class_weight="balanced"))]),
        construir_grid(config_hgb),
    ),
    "mlp_svd": (
        Pipeline([("prep", preprocesador_escalado), ("svd", TruncatedSVD(random_state=RANDOM_STATE)), ("clf", MLPClassifier(max_iter=120, early_stopping=True, random_state=RANDOM_STATE))]),
        construir_grid(config_mlp),
    ),
    "xgboost": (
        Pipeline([("prep", preprocesador_arboles), ("clf", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", tree_method="hist",
            device="cuda" if USAR_CUDA_XGBOOST else "cpu", random_state=RANDOM_STATE, n_jobs=-1,
        ))]),
        construir_grid(config_xgb),
    ),
}

resumen_configuraciones = pd.DataFrame([
    {"modelo": nombre, "configuraciones": len(param_grid)}
    for nombre, (_, param_grid) in modelos_parametros.items()
])
resumen_configuraciones["fits_cv_estimados"] = resumen_configuraciones["configuraciones"] * cv.get_n_splits()
resumen_configuraciones.to_csv(RUTA_RESULTADOS / f"00_configuraciones_por_modelo_v2_binario_{DATASET_USAR}.csv", index=False)
resumen_configuraciones


## 4. Validacion cruzada y entrenamiento

In [ ]:
resultados = []
mejores_modelos = {}
reportes = {}
cv_detallado = []
modelos_secuenciales = {"xgboost", "knn_svd", "mlp_svd", "gradient_boosting", "adaboost_tree"}

for nombre, (pipeline, params) in modelos_parametros.items():
    print(f"Evaluando {len(params)} configuraciones para: {nombre}")
    busqueda = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="f1_macro",
        cv=cv,
        n_jobs=1 if nombre in modelos_secuenciales else -1,
        verbose=1,
        error_score="raise",
        return_train_score=True,
    )
    try:
        if nombre == "xgboost":
            busqueda.fit(X_train, y_train, clf__sample_weight=pesos_train)
        else:
            busqueda.fit(X_train, y_train)
    except Exception as error:
        if nombre == "xgboost" and USAR_CUDA_XGBOOST:
            print("XGBoost CUDA fallo, reintentando en CPU:", error)
            pipeline.set_params(clf__device="cpu")
            busqueda = GridSearchCV(
                estimator=pipeline,
                param_grid=params,
                scoring="f1_macro",
                cv=cv,
                n_jobs=1,
                verbose=1,
                error_score="raise",
                return_train_score=True,
            )
            busqueda.fit(X_train, y_train, clf__sample_weight=pesos_train)
        else:
            raise

    df_cv = pd.DataFrame(busqueda.cv_results_)
    df_cv["modelo"] = nombre
    df_cv["dataset"] = DATASET_USAR
    df_cv.to_csv(RUTA_RESULTADOS / f"cv_detalle_{nombre}_v2_binario_{DATASET_USAR}.csv", index=False)
    cv_detallado.append(df_cv)

    mejor = busqueda.best_estimator_
    pred = mejor.predict(X_test)
    resultados.append({
        "dataset": DATASET_USAR,
        "modelo": nombre,
        "configuraciones_probadas": len(params),
        "fits_cv_estimados": len(params) * cv.get_n_splits(),
        "cv_f1_macro": busqueda.best_score_,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "test_f1_macro": f1_score(y_test, pred, average="macro"),
        "test_mse_auxiliar": mean_squared_error(y_test, pred),
        "mejores_parametros": json.dumps(busqueda.best_params_, ensure_ascii=False),
    })
    mejores_modelos[nombre] = mejor
    reportes[nombre] = classification_report(y_test, pred, target_names=le.classes_, output_dict=True, zero_division=0)

resultados_df = pd.DataFrame(resultados).sort_values("test_f1_macro", ascending=False)
cv_detallado_df = pd.concat(cv_detallado, ignore_index=True)

resultados_df.to_csv(RUTA_RESULTADOS / f"01_resultados_modelos_v2_binario_{DATASET_USAR}.csv", index=False)
cv_detallado_df.to_csv(RUTA_RESULTADOS / f"05_cv_detalle_todos_modelos_v2_binario_{DATASET_USAR}.csv", index=False)
with (RUTA_RESULTADOS / f"02_reportes_clasificacion_v2_binario_{DATASET_USAR}.json").open("w", encoding="utf-8") as f:
    json.dump(reportes, f, ensure_ascii=False, indent=2)
resultados_df


## 5. Resultado final, graficas y metricas clinicas

El mejor modelo base se selecciona por `test_f1_macro`, porque esta metrica penaliza que el modelo funcione bien solo en una clase. Para contexto clinico tambien se reportan sensibilidad, especificidad, precision, recall, ROC-AUC, PR-AUC, log loss, Brier score, MCC, kappa, MAE y MSE.

Ademas se calcula una tabla de overfitting usando la brecha `mean_train_score - mean_test_score` de validacion cruzada. Como mejora conservadora, se prueba ajuste de umbral en los 3 mejores modelos base usando una validacion interna separada del test.


In [ ]:
indice_resistant = int(np.where(le.classes_ == "Resistant")[0][0])
indice_susceptible = int(np.where(le.classes_ == "Susceptible")[0][0])


def obtener_proba_resistant(modelo, X_eval):
    """Devuelve probabilidad de Resistant si el modelo la soporta."""
    if not hasattr(modelo, "predict_proba"):
        return None
    proba = modelo.predict_proba(X_eval)
    clases_modelo = modelo.named_steps["clf"].classes_ if hasattr(modelo, "named_steps") else modelo.classes_
    pos = int(np.where(clases_modelo == indice_resistant)[0][0])
    return proba[:, pos]


def metricas_clinicas(nombre, modelo, X_eval, y_eval, umbral_resistant=None, etiqueta="base"):
    """Calcula metricas completas tratando Resistant como evento clinico principal."""
    proba_resistant = obtener_proba_resistant(modelo, X_eval)
    if umbral_resistant is not None and proba_resistant is not None:
        pred_eval = np.where(proba_resistant >= umbral_resistant, indice_resistant, indice_susceptible)
    else:
        pred_eval = modelo.predict(X_eval)

    y_bin = (np.asarray(y_eval) == indice_resistant).astype(int)
    pred_bin = (np.asarray(pred_eval) == indice_resistant).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_bin, pred_bin, labels=[0, 1]).ravel()

    sensibilidad_resistant = tp / (tp + fn) if (tp + fn) else np.nan
    especificidad_resistant = tn / (tn + fp) if (tn + fp) else np.nan
    ppv_resistant = tp / (tp + fp) if (tp + fp) else np.nan
    npv_resistant = tn / (tn + fn) if (tn + fn) else np.nan

    fila = {
        "modelo": nombre,
        "tipo_evaluacion": etiqueta,
        "umbral_resistant": 0.5 if umbral_resistant is None else umbral_resistant,
        "accuracy": accuracy_score(y_eval, pred_eval),
        "balanced_accuracy": balanced_accuracy_score(y_eval, pred_eval),
        "f1_macro": f1_score(y_eval, pred_eval, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_eval, pred_eval, average="weighted", zero_division=0),
        "precision_macro": precision_score(y_eval, pred_eval, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_eval, pred_eval, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_eval, pred_eval, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_eval, pred_eval, average="weighted", zero_division=0),
        "f2_resistant": fbeta_score(y_bin, pred_bin, beta=2, zero_division=0),
        "sensibilidad_resistant": sensibilidad_resistant,
        "especificidad_resistant": especificidad_resistant,
        "ppv_precision_resistant": ppv_resistant,
        "npv_resistant": npv_resistant,
        "mcc": matthews_corrcoef(y_eval, pred_eval),
        "cohen_kappa": cohen_kappa_score(y_eval, pred_eval),
        "mse": mean_squared_error(y_eval, pred_eval),
        "mae": mean_absolute_error(y_eval, pred_eval),
        "tn_susceptible_correcto": tn,
        "fp_resistant_falso": fp,
        "fn_resistant_no_detectado": fn,
        "tp_resistant_correcto": tp,
    }

    if proba_resistant is not None:
        eps = 1e-15
        proba_clip = np.clip(proba_resistant, eps, 1 - eps)
        fila.update({
            "roc_auc_resistant": roc_auc_score(y_bin, proba_resistant),
            "pr_auc_resistant": average_precision_score(y_bin, proba_resistant),
            "log_loss_resistant": log_loss(y_bin, proba_clip, labels=[0, 1]),
            "brier_score_resistant": brier_score_loss(y_bin, proba_resistant),
        })
    else:
        fila.update({
            "roc_auc_resistant": np.nan,
            "pr_auc_resistant": np.nan,
            "log_loss_resistant": np.nan,
            "brier_score_resistant": np.nan,
        })
    return fila


# 1) Metricas clinicas completas de todos los modelos base.
metricas_clinicas_base = pd.DataFrame([
    metricas_clinicas(nombre, modelo, X_test, y_test, etiqueta="base_umbral_0_5")
    for nombre, modelo in mejores_modelos.items()
]).sort_values("f1_macro", ascending=False)
metricas_clinicas_base.to_csv(RUTA_RESULTADOS / f"07_scores_clinicos_todos_modelos_v2_binario_{DATASET_USAR}.csv", index=False)


# Curvas ROC y Precision-Recall para todos los modelos base.
y_test_resistant_bin = (np.asarray(y_test) == indice_resistant).astype(int)
fig, ax = plt.subplots(figsize=(8, 6))
for nombre, modelo in mejores_modelos.items():
    proba_resistant = obtener_proba_resistant(modelo, X_test)
    if proba_resistant is None:
        continue
    fpr, tpr, _ = roc_curve(y_test_resistant_bin, proba_resistant)
    ax.plot(fpr, tpr, label=f"{nombre} AUC={auc(fpr, tpr):.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="#777777", label="azar")
ax.set_xlabel("Falsos positivos Resistant")
ax.set_ylabel("Verdaderos positivos Resistant")
ax.set_title("Curvas ROC - deteccion de Resistant")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"21_v2_binario_roc_todos_modelos_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for nombre, modelo in mejores_modelos.items():
    proba_resistant = obtener_proba_resistant(modelo, X_test)
    if proba_resistant is None:
        continue
    precision, recall, _ = precision_recall_curve(y_test_resistant_bin, proba_resistant)
    ax.plot(recall, precision, label=f"{nombre} AP={average_precision_score(y_test_resistant_bin, proba_resistant):.3f}")
ax.set_xlabel("Recall / sensibilidad Resistant")
ax.set_ylabel("Precision Resistant")
ax.set_title("Curvas Precision-Recall - deteccion de Resistant")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"22_v2_binario_pr_todos_modelos_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

# 2) Diagnostico de overfitting por CV.
cv_overfit_df = cv_detallado_df.copy()
cv_overfit_df["gap_train_test"] = cv_overfit_df["mean_train_score"] - cv_overfit_df["mean_test_score"]
mejores_cv_por_modelo = (
    cv_overfit_df.sort_values(["modelo", "mean_test_score"], ascending=[True, False])
    .groupby("modelo", as_index=False)
    .head(1)
    [["modelo", "mean_test_score", "std_test_score", "mean_train_score", "gap_train_test", "params"]]
    .sort_values("mean_test_score", ascending=False)
)
mejores_cv_por_modelo["riesgo_overfitting"] = pd.cut(
    mejores_cv_por_modelo["gap_train_test"],
    bins=[-np.inf, 0.03, 0.07, np.inf],
    labels=["bajo", "moderado", "alto"],
)
mejores_cv_por_modelo.to_csv(RUTA_RESULTADOS / f"08_diagnostico_overfitting_cv_v2_binario_{DATASET_USAR}.csv", index=False)

# 3) Mejora conservadora: ajuste de umbral solo con validacion interna del train.
X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)
umbrales = np.round(np.arange(0.25, 0.76, 0.01), 2)
modelos_top_umbral = resultados_df.head(3)["modelo"].tolist()
filas_umbral = []
for nombre in modelos_top_umbral:
    modelo_umbral = clone(mejores_modelos[nombre])
    if nombre == "xgboost":
        pesos_subtrain = compute_sample_weight(class_weight="balanced", y=y_subtrain)
        modelo_umbral.fit(X_subtrain, y_subtrain, clf__sample_weight=pesos_subtrain)
    else:
        modelo_umbral.fit(X_subtrain, y_subtrain)

    proba_val = obtener_proba_resistant(modelo_umbral, X_val)
    if proba_val is None:
        continue

    candidatos = []
    for umbral in umbrales:
        pred_val = np.where(proba_val >= umbral, indice_resistant, indice_susceptible)
        candidatos.append({
            "modelo": nombre,
            "umbral_resistant": umbral,
            "val_f1_macro": f1_score(y_val, pred_val, average="macro", zero_division=0),
            "val_balanced_accuracy": balanced_accuracy_score(y_val, pred_val),
            "val_mse": mean_squared_error(y_val, pred_val),
        })
    candidatos_df = pd.DataFrame(candidatos).sort_values(
        ["val_f1_macro", "val_balanced_accuracy", "val_mse"],
        ascending=[False, False, True],
    )
    mejor_umbral = float(candidatos_df.iloc[0]["umbral_resistant"])
    fila_test = metricas_clinicas(
        nombre,
        modelo_umbral,
        X_test,
        y_test,
        umbral_resistant=mejor_umbral,
        etiqueta="umbral_optimizado_validacion_interna",
    )
    fila_test.update({
        "val_f1_macro_umbral": float(candidatos_df.iloc[0]["val_f1_macro"]),
        "val_balanced_accuracy_umbral": float(candidatos_df.iloc[0]["val_balanced_accuracy"]),
        "val_mse_umbral": float(candidatos_df.iloc[0]["val_mse"]),
    })
    filas_umbral.append(fila_test)

metricas_umbral_df = pd.DataFrame(filas_umbral)
metricas_comparacion_df = pd.concat([metricas_clinicas_base, metricas_umbral_df], ignore_index=True, sort=False)
metricas_comparacion_df = metricas_comparacion_df.sort_values(
    ["f1_macro", "balanced_accuracy", "mse"], ascending=[False, False, True]
)
metricas_umbral_df.to_csv(RUTA_RESULTADOS / f"09_scores_umbral_optimizado_v2_binario_{DATASET_USAR}.csv", index=False)
metricas_comparacion_df.to_csv(RUTA_RESULTADOS / f"10_scores_clinicos_base_vs_umbral_v2_binario_{DATASET_USAR}.csv", index=False)

# 4) Mejor opcion operativa segun F1 macro, revisando tambien la brecha CV del modelo base.
mejor_operativo = metricas_comparacion_df.iloc[0].copy()
brecha_modelo_base = mejores_cv_por_modelo.set_index("modelo").loc[mejor_operativo["modelo"], "gap_train_test"]
riesgo_modelo_base = mejores_cv_por_modelo.set_index("modelo").loc[mejor_operativo["modelo"], "riesgo_overfitting"]

# 5) Matriz de confusion del mejor modelo base original.
mejor_nombre = resultados_df.iloc[0]["modelo"]
mejor_modelo = mejores_modelos[mejor_nombre]
pred = mejor_modelo.predict(X_test)
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"V2 binaria - mejor modelo base: {mejor_nombre}")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"13_v2_binario_matriz_confusion_{DATASET_USAR}_{mejor_nombre}.png", dpi=220, bbox_inches="tight")
plt.show()

# 6) Ranking de modelos base por F1 macro.
ranking_plot = resultados_df.sort_values("test_f1_macro", ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking_plot["modelo"], ranking_plot["test_f1_macro"], color="#315f72")
ax.set_xlabel("F1 macro en test")
ax.set_title("Ranking final de modelos base V2")
for i, valor in enumerate(ranking_plot["test_f1_macro"]):
    ax.text(valor + 0.002, i, f"{valor:.3f}", va="center")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"14_v2_binario_ranking_modelos_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

# 7) Distribucion de desempeno CV por modelo.
fig, ax = plt.subplots(figsize=(10, 5))
cv_detallado_df.boxplot(column="mean_test_score", by="modelo", ax=ax, rot=20)
ax.set_title("Distribucion F1 macro en validacion cruzada por modelo")
ax.set_xlabel("Modelo")
ax.set_ylabel("F1 macro CV")
fig.suptitle("")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"15_v2_binario_cv_boxplot_modelos_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

# 8) Metricas por organismo con el mejor modelo base.
analisis_org = X_test[["organism"]].copy() if "organism" in X_test.columns else pd.DataFrame({"organism": "SIN_ORGANISM"}, index=range(len(y_test)))
analisis_org["y_true"] = y_test
analisis_org["y_pred"] = pred
metricas_org = []
for organismo, grupo in analisis_org.groupby("organism"):
    metricas_org.append({
        "organism": organismo,
        "filas": len(grupo),
        "accuracy": accuracy_score(grupo["y_true"], grupo["y_pred"]),
        "balanced_accuracy": balanced_accuracy_score(grupo["y_true"], grupo["y_pred"]),
        "f1_macro": f1_score(grupo["y_true"], grupo["y_pred"], average="macro"),
        "mse_auxiliar": mean_squared_error(grupo["y_true"], grupo["y_pred"]),
    })
metricas_org = pd.DataFrame(metricas_org).sort_values("f1_macro", ascending=False)
metricas_org.to_csv(RUTA_RESULTADOS / f"03_metricas_por_organismo_v2_binario_{DATASET_USAR}.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
org_plot = metricas_org.sort_values("f1_macro", ascending=True)
ax.barh(org_plot["organism"], org_plot["f1_macro"], color="#8a5a44")
ax.set_xlabel("F1 macro en test")
ax.set_title(f"Desempeno por organismo - {mejor_nombre}")
for i, valor in enumerate(org_plot["f1_macro"]):
    ax.text(valor + 0.002, i, f"{valor:.3f}", va="center")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"16_v2_binario_metricas_por_organismo_{DATASET_USAR}_{mejor_nombre}.png", dpi=220, bbox_inches="tight")
plt.show()

# 9) Top configuraciones de validacion cruzada.
top_cv = cv_detallado_df.sort_values("mean_test_score", ascending=False).head(20).copy()
top_cv["config_label"] = top_cv["modelo"] + " #" + top_cv["rank_test_score"].astype(str)
fig, ax = plt.subplots(figsize=(10, 7))
plot_top = top_cv.sort_values("mean_test_score", ascending=True)
ax.barh(plot_top["config_label"], plot_top["mean_test_score"], color="#4d7c59")
ax.set_xlabel("F1 macro promedio CV")
ax.set_title("Top 20 configuraciones por validacion cruzada")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"17_v2_binario_top20_configuraciones_cv_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

# 10) Importancia de variables del mejor modelo base.
n_muestra_importancia = min(5000, len(X_test))
X_importancia = X_test.sample(n=n_muestra_importancia, random_state=RANDOM_STATE)
y_importancia = y_test.loc[X_importancia.index] if hasattr(y_test, "loc") else pd.Series(y_test, index=X_test.index).loc[X_importancia.index]
importancia = permutation_importance(
    mejor_modelo,
    X_importancia,
    y_importancia,
    n_repeats=3,
    random_state=RANDOM_STATE,
    scoring="f1_macro",
    n_jobs=1,
)
importancia_df = pd.DataFrame({
    "variable": X_test.columns,
    "importancia_media": importancia.importances_mean,
    "importancia_std": importancia.importances_std,
}).sort_values("importancia_media", ascending=False)
importancia_df.to_csv(RUTA_RESULTADOS / f"06_importancia_variables_mejor_modelo_v2_binario_{DATASET_USAR}.csv", index=False)

top_importancia = importancia_df.head(20).sort_values("importancia_media", ascending=True)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_importancia["variable"], top_importancia["importancia_media"], xerr=top_importancia["importancia_std"], color="#b05746")
ax.set_xlabel("Caida promedio de F1 macro al permutar")
ax.set_title(f"Top 20 variables mas importantes - {mejor_nombre}")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"18_v2_binario_importancia_variables_{DATASET_USAR}_{mejor_nombre}.png", dpi=220, bbox_inches="tight")
plt.show()

# 11) Grafica de brecha de overfitting CV.
fig, ax = plt.subplots(figsize=(9, 5))
gap_plot = mejores_cv_por_modelo.sort_values("gap_train_test", ascending=True)
ax.barh(gap_plot["modelo"], gap_plot["gap_train_test"], color="#9a6b2f")
ax.axvline(0.03, color="#3f7d5c", linestyle="--", linewidth=1, label="bajo/moderado")
ax.axvline(0.07, color="#a33f3f", linestyle="--", linewidth=1, label="moderado/alto")
ax.set_xlabel("Brecha CV train - validacion")
ax.set_title("Diagnostico de overfitting por modelo")
ax.legend()
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"19_v2_binario_overfitting_gap_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

# 12) Comparacion base vs umbral optimizado.
fig, ax = plt.subplots(figsize=(11, 6))
plot_comp = metricas_comparacion_df.head(10).copy()
plot_comp["label"] = plot_comp["modelo"] + " - " + plot_comp["tipo_evaluacion"]
plot_comp = plot_comp.sort_values("f1_macro", ascending=True)
ax.barh(plot_comp["label"], plot_comp["f1_macro"], color="#2f6f73")
ax.set_xlabel("F1 macro en test")
ax.set_title("Comparacion base vs umbral optimizado en validacion interna")
for i, valor in enumerate(plot_comp["f1_macro"]):
    ax.text(valor + 0.002, i, f"{valor:.3f}", va="center")
fig.tight_layout()
fig.savefig(RUTA_GRAFICAS / f"20_v2_binario_base_vs_umbral_{DATASET_USAR}.png", dpi=220, bbox_inches="tight")
plt.show()

resumen_final = pd.DataFrame({
    "dataset": [DATASET_USAR],
    "mejor_modelo_base": [mejor_nombre],
    "mejor_opcion_operativa": [mejor_operativo["modelo"]],
    "tipo_evaluacion_operativa": [mejor_operativo["tipo_evaluacion"]],
    "criterio_seleccion": ["mayor f1_macro en test; umbral elegido solo con validacion interna cuando aplica"],
    "umbral_resistant": [mejor_operativo["umbral_resistant"]],
    "test_f1_macro": [mejor_operativo["f1_macro"]],
    "test_balanced_accuracy": [mejor_operativo["balanced_accuracy"]],
    "test_accuracy": [mejor_operativo["accuracy"]],
    "test_sensibilidad_resistant": [mejor_operativo["sensibilidad_resistant"]],
    "test_especificidad_resistant": [mejor_operativo["especificidad_resistant"]],
    "test_mse": [mejor_operativo["mse"]],
    "test_roc_auc_resistant": [mejor_operativo["roc_auc_resistant"]],
    "test_pr_auc_resistant": [mejor_operativo["pr_auc_resistant"]],
    "gap_overfitting_cv_modelo_base": [brecha_modelo_base],
    "riesgo_overfitting_modelo_base": [str(riesgo_modelo_base)],
    "configuraciones_totales": [int(resultados_df["configuraciones_probadas"].sum())],
    "fits_cv_estimados": [int(resultados_df["fits_cv_estimados"].sum())],
    "mejores_parametros_base": [resultados_df.loc[resultados_df["modelo"].eq(mejor_nombre), "mejores_parametros"].iloc[0]],
})
resumen_final.to_csv(RUTA_RESULTADOS / f"04_mejor_modelo_v2_binario_{DATASET_USAR}.csv", index=False)

print("MEJOR MODELO BASE:", mejor_nombre)
print("MEJOR OPCION OPERATIVA:", mejor_operativo["modelo"], "|", mejor_operativo["tipo_evaluacion"])
print("Umbral Resistant:", round(float(mejor_operativo["umbral_resistant"]), 3))
print("F1 macro test:", round(float(mejor_operativo["f1_macro"]), 4))
print("Balanced accuracy test:", round(float(mejor_operativo["balanced_accuracy"]), 4))
print("Sensibilidad Resistant:", round(float(mejor_operativo["sensibilidad_resistant"]), 4))
print("Especificidad Resistant:", round(float(mejor_operativo["especificidad_resistant"]), 4))
print("MSE test:", round(float(mejor_operativo["mse"]), 4))
print("Riesgo overfitting CV del modelo base:", str(riesgo_modelo_base), "| gap:", round(float(brecha_modelo_base), 4))

resumen_final


## 6. Cierre

La seleccion final del modelo se hace con `f1_macro` en test, pero para contexto clinico tambien se deben mirar sensibilidad de `Resistant`, especificidad, ROC-AUC, PR-AUC, precision, recall, MCC, kappa y MSE.

El MSE se reporta como metrica auxiliar porque este problema es de clasificacion, no de regresion. En clasificacion binaria con etiquetas 0/1, el MSE equivale a la proporcion de errores cuando se calcula sobre clases predichas.

Archivos principales que deja este notebook:

- `01_resultados_modelos_v2_binario_balanceado_organismo_clase.csv`: ranking final por modelo base.
- `05_cv_detalle_todos_modelos_v2_binario_balanceado_organismo_clase.csv`: detalle de las 2800 configuraciones evaluadas.
- `03_metricas_por_organismo_v2_binario_balanceado_organismo_clase.csv`: desempeno del mejor modelo base por bacteria.
- `04_mejor_modelo_v2_binario_balanceado_organismo_clase.csv`: resumen final de la opcion operativa recomendada.
- `06_importancia_variables_mejor_modelo_v2_binario_balanceado_organismo_clase.csv`: variables mas influyentes segun importancia por permutacion.
- `07_scores_clinicos_todos_modelos_v2_binario_balanceado_organismo_clase.csv`: scores clinicos completos de todos los modelos base.
- `08_diagnostico_overfitting_cv_v2_binario_balanceado_organismo_clase.csv`: brecha train-validacion para revisar sobreentrenamiento.
- `09_scores_umbral_optimizado_v2_binario_balanceado_organismo_clase.csv`: resultados de ajuste conservador de umbral.
- `10_scores_clinicos_base_vs_umbral_v2_binario_balanceado_organismo_clase.csv`: comparacion final entre modelos base y modelos con umbral ajustado.

Graficas nuevas o actualizadas:

- `13`: matriz de confusion del mejor modelo base.
- `14`: ranking final de modelos base.
- `15`: distribucion de F1 macro en validacion cruzada.
- `16`: desempeno por organismo.
- `17`: top 20 configuraciones CV.
- `18`: top 20 variables importantes.
- `19`: brecha de overfitting por modelo.
- `20`: comparacion base vs umbral optimizado.
- `21`: curvas ROC para todos los modelos base.
- `22`: curvas precision-recall para todos los modelos base.
